# Spam-Ham Sınıflandırması TF-IDF ve Naive Bayes ile

Bu çalışmada:
- Metin verisi üzerinde temizlik işlemleri yapılacaktır.
- TF-IDF vektörleştirme yöntemi kullanılacaktır.
- Naive Bayes algoritması ile sınıflandırma yapılacaktır.
- Model başarımını artırmak için çeşitli iyileştirmeler uygulanacaktır.


In [2]:
import pandas as pd
import string
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

nltk.download("stopwords")
nltk.download("wordnet")

# Veri yükleme
df = pd.read_csv("spam_ham_dataset.csv")
df = df[["text", "label_num"]]
df.head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\misla\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\misla\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,text,label_num
0,Subject: enron methanol ; meter # : 988291\nth...,0
1,"Subject: hpl nom for january 9 , 2001\n( see a...",0
2,"Subject: neon retreat\nho ho ho , we ' re arou...",0
3,"Subject: photoshop , windows , office . cheap ...",1
4,Subject: re : indian springs\nthis deal is to ...,0


# TF-IDF Vectorizer Nedir?

TF-IDF, bir kelimenin belirli bir belge için ne kadar önemli olduğunu ölçen bir tekniktir.

- **TF (Term Frequency):** Kelimenin belgede kaç kere geçtiğidir.
- **IDF (Inverse Document Frequency):** Kelimenin tüm belgelerde ne kadar nadir geçtiğini ölçer.

TFIDF, bu iki değerin çarpımıdır.

\[
TFIDF(t, d) = TF(t, d) \times \log\left(\frac{N}{df(t)}\right)
\]

TF-IDF sayesinde daha anlamlı kelimeler modele katkı sağlar.

In [12]:
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"subject|fw|re", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    words = [word for word in words if not word.isdigit()]
    words = [lemmatizer.lemmatize(word) for word in words if word not in stopwords.words("english")]
    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head()

,text,clean_text
0,Subject: enron methanol ; meter # : 988291\nth...,enron methanol meter follow note gave monday p...
1,"Subject: hpl nom for january 9 , 2001\n( see a...",hpl nom january see attached file hplnol xl hp...
2,"Subject: neon retreat\nho ho ho , we ' re arou...",neon tat ho ho ho around wonderful time year n...
3,"Subject: photoshop , windows , office . cheap ...",photoshop window office cheap main tnding abas...
4,Subject: re : indian springs\nthis deal is to ...,indian spring deal book teco pvr venue underst...


In [16]:
# TF-IDF (1-gram ve 2-gram ile)
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, stop_words="english")
X_tfidf = tfidf.fit_transform(df["clean_text"])
y = df["label_num"]

# Veri bölme
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.3, random_state=42)

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
